# 09 — Especificidad del baseline y complejidad de la trayectoria

## Proyecto
**Kaki_DeepHS_StorageDays**

## Pregunta central

El notebook 08 mostró que, para VIS_COR, la representación longitudinal:

\[
\Delta R_{i,t}=R_{i,t}-R_{i,0}
\]

especialmente después de SNV, mejora de manera importante la estimación del tiempo de almacenamiento.

Ahora necesitamos responder dos preguntas críticas antes de cerrar la fase principal de modelado clásico.

### Pregunta 1 — ¿Importa que el baseline sea del mismo fruto?

Se comparan:

1. **Self baseline**
   \[
   R_{i,t}-R_{i,0}
   \]

2. **Training-mean baseline**
   \[
   R_{i,t}-\overline{R_{0,\text{training}}}
   \]

3. **Random-other baseline**
   \[
   R_{i,t}-R_{j,0},
   \qquad j\neq i
   \]

El baseline promedio y los baselines aleatorios de control se construyen usando exclusivamente frutos de training en cada fold.

### Pregunta 2 — ¿La predicción usa la forma espectral multivariada o solo la distancia al baseline?

Se comparan:

- Ridge multibanda sobre \(\Delta SNV\).
- RMS de \(\Delta SNV\).
- norma L2 de \(\Delta SNV\).
- spectral angle entre \(R_t\) y \(R_0\).
- combinación escalar `RMS + spectral angle`.
- DummyMedian.

Si la representación multibanda supera claramente a estos controles escalares, la información temporal no se explica únicamente por "qué tan lejos" está el fruto respecto al día 0.

## Sensores

Se ejecuta para:

- VIS
- VIS_COR

pero el foco interpretativo principal sigue siendo VIS_COR.

## Validación

- outer CV congelada, fruit-disjoint;
- inner CV congelada, fruit-disjoint;
- tuning de Ridge solo dentro de development;
- bootstrap pareado por fruto;
- stress test `session-excluded` con null sometido a la misma exclusión.

## Limitación

Los modelos con `self baseline` requieren una medición del mismo fruto en día 0. Son controles longitudinales y no modelos de una sola medición.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import platform
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "09_kaki_baseline_specificity_trajectory_complexity"
RUN_SIGNATURE = "baseline_specificity_trajectory_complexity_v1"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

BASE08 = ROOT / "results/08_kaki_within_fruit_delta_trajectory"

RESULTS = ROOT / "results/09_kaki_baseline_specificity_trajectory_complexity"
FIGURES = ROOT / "figures/09_kaki_baseline_specificity_trajectory_complexity"
TABLES = ROOT / "tables/09_kaki_baseline_specificity_trajectory_complexity"
EXPORT_ROOT = ROOT / "exports/09_kaki_baseline_specificity_trajectory_complexity"

for p in [
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

CAMERAS = [
    "VIS",
    "VIS_COR",
]

RIDGE_ALPHAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]

OUTER_FOLDS = 5
INNER_FOLDS = 4

RANDOM_SEED = 20260818
BOOTSTRAP_REPS = 5000
N_RANDOM_BASELINE_REPS = 10

print("ROOT:", ROOT)
print("Random-other baseline reps:", N_RANDOM_BASELINE_REPS)


## 2. Cargar datos y splits congelados


In [ ]:
SPECTRA_PATHS = {
    "VIS":
        INTERIM /
        "kaki_fruit_day_spectra_VIS.csv",

    "VIS_COR":
        INTERIM /
        "kaki_fruit_day_spectra_VIS_COR.csv",
}

CUBE_PATHS = {
    "VIS":
        INTERIM /
        "kaki_cube_spectra_VIS.csv",

    "VIS_COR":
        INTERIM /
        "kaki_cube_spectra_VIS_COR.csv",
}

OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

DELTA08_OOF_PATH = (
    BASE08 /
    "kaki_delta_standard_oof.csv"
)

DELTA08_SESSION_PATH = (
    BASE08 /
    "kaki_delta_session_excluded_oof.csv"
)

for p in [
    *SPECTRA_PATHS.values(),
    *CUBE_PATHS.values(),
    OUTER_SPLIT_PATH,
    INNER_SPLIT_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(
            f"Falta archivo requerido: {p}"
        )

spectra = {}
wavelengths = {}

expected_cube_counts = {
    "VIS": 395,
    "VIS_COR": 398,
}

qc_rows = []

for camera in CAMERAS:

    cube = pd.read_csv(
        CUBE_PATHS[
            camera
        ]
    )

    qc_rows.append({
        "camera_type":
            camera,

        "n_cubes":
            int(
                len(
                    cube
                )
            ),

        "median_mask_coverage":
            float(
                cube[
                    "mask_coverage"
                ].median()
            ),

        "n_qc_not_ok":
            int(
                (
                    cube[
                        "mask_qc_flag"
                    ]
                    !=
                    "ok"
                ).sum()
            ),

        "n_extraction_errors":
            int(
                cube[
                    "extraction_error"
                ].notna().sum()
            ),
    })

    assert len(cube) == expected_cube_counts[camera]
    assert float(cube["mask_coverage"].median()) < 0.87
    assert (cube["mask_qc_flag"] == "ok").all()
    assert cube["extraction_error"].isna().all()

    df = pd.read_csv(
        SPECTRA_PATHS[
            camera
        ]
    )

    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

    df[
        "storage_days_model"
    ] = pd.to_numeric(
        df[
            "storage_days_model"
        ],
        errors="raise"
    ).astype(int)

    wl_cols = [
        c
        for c in df.columns
        if c.startswith("wl_")
    ]

    wl = np.array(
        [
            float(
                c.replace(
                    "wl_",
                    ""
                )
            )
            for c in wl_cols
        ],
        dtype=float
    )

    assert len(df) == 199
    assert df["fruit_id"].nunique() == 40
    assert not df[wl_cols].isna().any().any()

    spectra[camera] = (
        df.sort_values(
            [
                "fruit_id",
                "storage_days_model"
            ]
        )
        .reset_index(drop=True)
    )

    wavelengths[camera] = wl

qc = pd.DataFrame(qc_rows)
display(qc)

outer = pd.read_csv(
    OUTER_SPLIT_PATH
)

inner = pd.read_csv(
    INNER_SPLIT_PATH
)

for df in [
    outer,
    inner,
]:

    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

outer[
    "outer_fold"
] = pd.to_numeric(
    outer[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "outer_fold"
] = pd.to_numeric(
    inner[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "inner_fold"
] = pd.to_numeric(
    inner[
        "inner_fold"
    ],
    errors="raise"
).astype(int)

print("✅ Inputs cargados.")


## 3. Auditoría de splits


In [ ]:
all_fruits = set(
    outer[
        "fruit_id"
    ].astype(int)
)

audit_rows = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    dev_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner.loc[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    audit_rows.append({
        "outer_fold":
            int(
                outer_fold
            ),

        "n_dev_fruits":
            len(
                dev_fruits
            ),

        "n_test_fruits":
            len(
                test_fruits
            ),

        "train_test_overlap":
            len(
                dev_fruits
                &
                test_fruits
            ),

        "inner_test_overlap":
            len(
                inner_fruits
                &
                test_fruits
            ),

        "inner_matches_dev":
            inner_fruits
            ==
            dev_fruits,
    })

split_audit = pd.DataFrame(
    audit_rows
)

display(
    split_audit
)

assert (
    split_audit[
        "train_test_overlap"
    ]
    ==
    0
).all()

assert (
    split_audit[
        "inner_test_overlap"
    ]
    ==
    0
).all()

assert split_audit[
    "inner_matches_dev"
].all()

split_audit.to_csv(
    TABLES /
    "01_split_audit.csv",
    index=False
)

print("✅ Splits válidos.")


# 4. Utilidades espectrales


In [ ]:
def snv_rows(
    X
):

    X = np.asarray(
        X,
        dtype=float
    )

    mu = X.mean(
        axis=1,
        keepdims=True
    )

    sd = X.std(
        axis=1,
        keepdims=True
    )

    sd[
        (~np.isfinite(sd))
        |
        (sd == 0)
    ] = 1.0

    return (
        X
        -
        mu
    ) / sd


def spectral_angle_rows(
    A,
    B,
    eps=1e-12
):

    A = np.asarray(
        A,
        dtype=float
    )

    B = np.asarray(
        B,
        dtype=float
    )

    num = np.sum(
        A
        *
        B,
        axis=1
    )

    den = (
        np.linalg.norm(
            A,
            axis=1
        )
        *
        np.linalg.norm(
            B,
            axis=1
        )
    )

    cosang = np.divide(
        num,
        den,
        out=np.ones_like(
            num,
            dtype=float
        ),
        where=den > eps
    )

    cosang = np.clip(
        cosang,
        -1.0,
        1.0
    )

    return np.arccos(
        cosang
    )


def fit_predict_ridge(
    X_train,
    y_train,
    X_test,
    alpha
):

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(
        X_train
    )

    X_test_s = scaler.transform(
        X_test
    )

    model = Ridge(
        alpha=float(
            alpha
        )
    )

    model.fit(
        X_train_s,
        y_train
    )

    return np.asarray(
        model.predict(
            X_test_s
        ),
        dtype=float
    )


# 5. Construcción de datasets por tipo de baseline

Todas las tareas predictivas excluyen día 0.

Se construyen:

- `self_delta_snv`
- `trainmean_delta_snv`
- `randomother_delta_snv`
- `self_scalar_rms`
- `self_scalar_l2`
- `self_scalar_angle`
- `self_scalar_rms_angle`

El baseline `training-mean` y los `random-other` deben recalcularse dentro de cada fold; por eso no se construyen como una tabla global fija.


In [ ]:
def split_day0_and_later(
    df
):

    wl_cols = [
        c
        for c in df.columns
        if c.startswith(
            "wl_"
        )
    ]

    baseline = (
        df[
            df[
                "storage_days_model"
            ]
            ==
            0
        ][
            [
                "fruit_id",
                *wl_cols
            ]
        ]
        .set_index(
            "fruit_id"
        )
        .sort_index()
    )

    later = (
        df[
            df[
                "storage_days_model"
            ]
            >
            0
        ]
        .copy()
        .sort_values(
            [
                "fruit_id",
                "storage_days_model"
            ]
        )
        .reset_index(
            drop=True
        )
    )

    assert len(
        baseline
    ) == 40

    assert len(
        later
    ) == 159

    return (
        baseline,
        later,
        wl_cols
    )


baseline_tables = {}
later_tables = {}
spectral_columns = {}

for camera in CAMERAS:

    b, l, cols = split_day0_and_later(
        spectra[
            camera
        ]
    )

    baseline_tables[
        camera
    ] = b

    later_tables[
        camera
    ] = l

    spectral_columns[
        camera
    ] = cols

print("✅ Baselines día 0 listos.")


# 6. Función de features con baseline propio


In [ ]:
def make_self_features(
    camera,
    fruit_ids
):

    baseline = baseline_tables[
        camera
    ]

    later = later_tables[
        camera
    ]

    cols = spectral_columns[
        camera
    ]

    d = (
        later[
            later[
                "fruit_id"
            ].isin(
                fruit_ids
            )
        ]
        .copy()
        .sort_values(
            [
                "fruit_id",
                "storage_days_model"
            ]
        )
        .reset_index(
            drop=True
        )
    )

    if len(
        d
    ) == 0:

        return d

    X = d[
        cols
    ].to_numpy(
        dtype=float
    )

    X0 = np.vstack(
        [
            baseline.loc[
                int(
                    fruit_id
                ),
                cols
            ].to_numpy(
                dtype=float
            )
            for fruit_id in d[
                "fruit_id"
            ]
        ]
    )

    X_snv = snv_rows(
        X
    )

    X0_snv = snv_rows(
        X0
    )

    delta_snv = (
        X_snv
        -
        X0_snv
    )

    rms = np.sqrt(
        np.mean(
            delta_snv ** 2,
            axis=1
        )
    )

    l2 = np.linalg.norm(
        delta_snv,
        axis=1
    )

    angle = spectral_angle_rows(
        X_snv,
        X0_snv
    )

    meta = d[
        [
            "fruit_id",
            "storage_days_model",
            "day_code",
        ]
    ].copy()

    return {
        "meta":
            meta,

        "delta_snv":
            delta_snv,

        "rms":
            rms.reshape(
                -1,
                1
            ),

        "l2":
            l2.reshape(
                -1,
                1
            ),

        "angle":
            angle.reshape(
                -1,
                1
            ),

        "rms_angle":
            np.column_stack(
                [
                    rms,
                    angle
                ]
            ),

        "current_snv":
            X_snv,

        "baseline_snv":
            X0_snv,
    }


# 7. Baseline promedio de training


In [ ]:
def training_mean_baseline_snv(
    camera,
    training_fruits
):

    baseline = baseline_tables[
        camera
    ]

    cols = spectral_columns[
        camera
    ]

    B = baseline.loc[
        sorted(
            training_fruits
        ),
        cols
    ].to_numpy(
        dtype=float
    )

    B_snv = snv_rows(
        B
    )

    return np.mean(
        B_snv,
        axis=0
    )


def make_trainmean_delta_features(
    camera,
    rows_fruits,
    baseline_training_fruits
):

    later = later_tables[
        camera
    ]

    cols = spectral_columns[
        camera
    ]

    d = (
        later[
            later[
                "fruit_id"
            ].isin(
                rows_fruits
            )
        ]
        .copy()
        .sort_values(
            [
                "fruit_id",
                "storage_days_model"
            ]
        )
        .reset_index(
            drop=True
        )
    )

    X = snv_rows(
        d[
            cols
        ].to_numpy(
            dtype=float
        )
    )

    mean_b = training_mean_baseline_snv(
        camera,
        baseline_training_fruits
    )

    delta = (
        X
        -
        mean_b[
            None,
            :
        ]
    )

    return {
        "meta":
            d[
                [
                    "fruit_id",
                    "storage_days_model",
                    "day_code",
                ]
            ].copy(),

        "X":
            delta,
    }


# 8. Baseline aleatorio de otro fruto

Para cada observación se asigna un baseline día 0 de un fruto distinto perteneciente al conjunto permitido de training.

Se usan 10 repeticiones deterministas.


In [ ]:
def make_randomother_delta_features(
    camera,
    rows_fruits,
    donor_training_fruits,
    rep,
    seed=RANDOM_SEED
):

    later = later_tables[
        camera
    ]

    baseline = baseline_tables[
        camera
    ]

    cols = spectral_columns[
        camera
    ]

    d = (
        later[
            later[
                "fruit_id"
            ].isin(
                rows_fruits
            )
        ]
        .copy()
        .sort_values(
            [
                "fruit_id",
                "storage_days_model"
            ]
        )
        .reset_index(
            drop=True
        )
    )

    X = snv_rows(
        d[
            cols
        ].to_numpy(
            dtype=float
        )
    )

    donor_ids = sorted(
        set(
            donor_training_fruits
        )
    )

    donor_baselines = snv_rows(
        baseline.loc[
            donor_ids,
            cols
        ].to_numpy(
            dtype=float
        )
    )

    donor_map = {
        int(fid):
            donor_baselines[
                idx
            ]
        for idx, fid in enumerate(
            donor_ids
        )
    }

    rng = np.random.default_rng(
        seed
        +
        int(
            rep
        )
        *
        10007
        +
        len(
            donor_ids
        )
        *
        97
    )

    chosen = []

    for fruit_id in d[
        "fruit_id"
    ].astype(int):

        eligible = [
            fid
            for fid in donor_ids
            if fid != fruit_id
        ]

        # Si el fruit_id no está en donor_ids, todos los donors son válidos.
        if not eligible:

            eligible = donor_ids

        donor = int(
            rng.choice(
                eligible
            )
        )

        chosen.append(
            donor_map[
                donor
            ]
        )

    B = np.vstack(
        chosen
    )

    delta = (
        X
        -
        B
    )

    return {
        "meta":
            d[
                [
                    "fruit_id",
                    "storage_days_model",
                    "day_code",
                ]
            ].copy(),

        "X":
            delta,
    }


# 9. Métricas


In [ ]:
def fruit_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def day_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "y_true"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def metric_bundle(
    df
):

    y = df[
        "y_true"
    ].to_numpy(
        dtype=float
    )

    p = df[
        "y_pred"
    ].to_numpy(
        dtype=float
    )

    residual = (
        p
        -
        y
    )

    rho, _ = spearmanr(
        y,
        p
    )

    return {
        "n_observations":
            int(
                len(
                    df
                )
            ),

        "n_fruits":
            int(
                df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y,
                    p
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y,
                        p
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y,
                    p
                )
            ),

        "spearman_rho":
            float(
                rho
            ),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <=
                    1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


# 10. CV interna para baseline propio y controles escalares


In [ ]:
def inner_select_self_feature(
    camera,
    development_fruits,
    inner_assignment,
    feature_name
):

    best = None

    rows = []

    for alpha in RIDGE_ALPHAS:

        parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_fruits = (
                set(
                    development_fruits
                )
                -
                val_fruits
            )

            tr = make_self_features(
                camera,
                train_fruits
            )

            va = make_self_features(
                camera,
                val_fruits
            )

            if len(
                va[
                    "meta"
                ]
            ) == 0:
                continue

            pred = fit_predict_ridge(
                tr[
                    feature_name
                ],
                tr[
                    "meta"
                ][
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                va[
                    feature_name
                ],
                alpha
            )

            parts.append(
                pd.DataFrame({
                    "fruit_id":
                        va[
                            "meta"
                        ][
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        va[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        p = pd.concat(
            parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,
            }

    return best


# 11. CV interna para training-mean baseline


In [ ]:
def inner_select_trainmean(
    camera,
    development_fruits,
    inner_assignment
):

    best = None

    rows = []

    for alpha in RIDGE_ALPHAS:

        parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_fruits = (
                set(
                    development_fruits
                )
                -
                val_fruits
            )

            tr = make_trainmean_delta_features(
                camera,
                rows_fruits=train_fruits,
                baseline_training_fruits=train_fruits
            )

            va = make_trainmean_delta_features(
                camera,
                rows_fruits=val_fruits,
                baseline_training_fruits=train_fruits
            )

            if len(
                va[
                    "meta"
                ]
            ) == 0:
                continue

            pred = fit_predict_ridge(
                tr[
                    "X"
                ],
                tr[
                    "meta"
                ][
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                va[
                    "X"
                ],
                alpha
            )

            parts.append(
                pd.DataFrame({
                    "fruit_id":
                        va[
                            "meta"
                        ][
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        va[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        p = pd.concat(
            parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        rank = (
            fb,
            db,
            mae,
            float(
                alpha
            ),
        )

        rows.append({
            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,
            }

    return best


# 12. OOF estándar — baseline propio, training mean y controles escalares


In [ ]:
self_feature_map = {
    "self_multiband_delta_snv":
        "delta_snv",

    "self_scalar_rms":
        "rms",

    "self_scalar_l2":
        "l2",

    "self_scalar_angle":
        "angle",

    "self_scalar_rms_angle":
        "rms_angle",
}

standard_parts = []
selected_rows = []

for camera in CAMERAS:

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            all_fruits
            -
            test_fruits
        )

        inner_assignment = inner[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold
        ][
            [
                "fruit_id",
                "inner_fold",
            ]
        ].copy()

        for model_name, feature_name in self_feature_map.items():

            best = inner_select_self_feature(
                camera,
                development_fruits,
                inner_assignment,
                feature_name
            )

            tr = make_self_features(
                camera,
                development_fruits
            )

            te = make_self_features(
                camera,
                test_fruits
            )

            if len(
                te[
                    "meta"
                ]
            ) == 0:
                continue

            pred = fit_predict_ridge(
                tr[
                    feature_name
                ],
                tr[
                    "meta"
                ][
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                te[
                    feature_name
                ],
                best[
                    "alpha"
                ]
            )

            standard_parts.append(
                pd.DataFrame({
                    "camera_type":
                        camera,

                    "protocol":
                        "standard",

                    "model":
                        model_name,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "alpha":
                        float(
                            best[
                                "alpha"
                            ]
                        ),

                    "fruit_id":
                        te[
                            "meta"
                        ][
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        te[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        te[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

            selected_rows.append({
                "camera_type":
                    camera,

                "protocol":
                    "standard",

                "model":
                    model_name,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "alpha":
                    float(
                        best[
                            "alpha"
                        ]
                    ),
            })

        best_mean = inner_select_trainmean(
            camera,
            development_fruits,
            inner_assignment
        )

        trm = make_trainmean_delta_features(
            camera,
            rows_fruits=development_fruits,
            baseline_training_fruits=development_fruits
        )

        tem = make_trainmean_delta_features(
            camera,
            rows_fruits=test_fruits,
            baseline_training_fruits=development_fruits
        )

        pred_mean = fit_predict_ridge(
            trm[
                "X"
            ],
            trm[
                "meta"
            ][
                "storage_days_model"
            ].to_numpy(
                dtype=float
            ),
            tem[
                "X"
            ],
            best_mean[
                "alpha"
            ]
        )

        standard_parts.append(
            pd.DataFrame({
                "camera_type":
                    camera,

                "protocol":
                    "standard",

                "model":
                    "training_mean_multiband_delta_snv",

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "alpha":
                    float(
                        best_mean[
                            "alpha"
                        ]
                    ),

                "fruit_id":
                    tem[
                        "meta"
                    ][
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    tem[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    tem[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    pred_mean,
            })
        )

        selected_rows.append({
            "camera_type":
                camera,

            "protocol":
                "standard",

            "model":
                "training_mean_multiband_delta_snv",

            "outer_fold":
                int(
                    outer_fold
                ),

            "alpha":
                float(
                    best_mean[
                        "alpha"
                    ]
                ),
        })

standard_oof = pd.concat(
    standard_parts,
    ignore_index=True
)

selected_configs = pd.DataFrame(
    selected_rows
)

for camera in CAMERAS:

    for model_name in list(
        self_feature_map.keys()
    ) + [
        "training_mean_multiband_delta_snv"
    ]:

        g = standard_oof[
            (
                standard_oof[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                standard_oof[
                    "model"
                ]
                ==
                model_name
            )
        ]

        assert len(
            g
        ) == 159

print("✅ OOF estándar completado.")


# 13. Random-other baseline — control permutacional

Para mantener el costo computacional razonable:

- el alpha se toma del `training_mean_multiband_delta_snv` del mismo outer fold;
- se ejecutan 10 asignaciones aleatorias independientes de baseline;
- cada asignación utiliza solo baselines día 0 de frutos de development.

Este control no busca optimizar el mejor modelo random; busca estimar qué ocurre cuando se destruye la correspondencia longitudinal correcta.


In [ ]:
random_parts = []

for camera in CAMERAS:

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            all_fruits
            -
            test_fruits
        )

        alpha = float(
            selected_configs.loc[
                (
                    selected_configs[
                        "camera_type"
                    ]
                    ==
                    camera
                )
                &
                (
                    selected_configs[
                        "protocol"
                    ]
                    ==
                    "standard"
                )
                &
                (
                    selected_configs[
                        "model"
                    ]
                    ==
                    "training_mean_multiband_delta_snv"
                )
                &
                (
                    selected_configs[
                        "outer_fold"
                    ]
                    ==
                    outer_fold
                ),
                "alpha"
            ].iloc[
                0
            ]
        )

        for rep in range(
            N_RANDOM_BASELINE_REPS
        ):

            tr = make_randomother_delta_features(
                camera,
                rows_fruits=development_fruits,
                donor_training_fruits=development_fruits,
                rep=rep
            )

            te = make_randomother_delta_features(
                camera,
                rows_fruits=test_fruits,
                donor_training_fruits=development_fruits,
                rep=rep
            )

            pred = fit_predict_ridge(
                tr[
                    "X"
                ],
                tr[
                    "meta"
                ][
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                te[
                    "X"
                ],
                alpha
            )

            random_parts.append(
                pd.DataFrame({
                    "camera_type":
                        camera,

                    "protocol":
                        "standard",

                    "model":
                        "random_other_multiband_delta_snv",

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "rep":
                        int(
                            rep
                        ),

                    "alpha":
                        alpha,

                    "fruit_id":
                        te[
                            "meta"
                        ][
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        te[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        te[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

random_oof = pd.concat(
    random_parts,
    ignore_index=True
)

assert len(
    random_oof
) == (
    2
    *
    159
    *
    N_RANDOM_BASELINE_REPS
)

print("✅ Random-other permutations completas.")


# 14. Null estándar de la tarea días > 0


In [ ]:
standard_null_parts = []

reference = later_tables[
    "VIS_COR"
][
    [
        "fruit_id",
        "storage_days_model",
    ]
].copy()

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    tr = reference[
        ~reference[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    te = reference[
        reference[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    null_value = float(
        np.median(
            tr[
                "storage_days_model"
            ]
        )
    )

    standard_null_parts.append(
        pd.DataFrame({
            "outer_fold":
                int(
                    outer_fold
                ),

            "fruit_id":
                te[
                    "fruit_id"
                ].to_numpy(
                    dtype=int
                ),

            "storage_days":
                te[
                    "storage_days_model"
                ].to_numpy(
                    dtype=int
                ),

            "y_true":
                te[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),

            "y_pred":
                np.repeat(
                    null_value,
                    len(
                        te
                    )
                ),
        })
    )

standard_null = pd.concat(
    standard_null_parts,
    ignore_index=True
)

assert len(
    standard_null
) == 159

standard_null_metrics = metric_bundle(
    standard_null
)

print(
    "DummyMedian FB-MAE:",
    standard_null_metrics[
        "fruit_balanced_mae"
    ]
)


# 15. Métricas estándar


In [ ]:
metric_rows = []

for (
    camera,
    model
), g in standard_oof.groupby(
    [
        "camera_type",
        "model",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "model":
            model,

        "protocol":
            "standard",
    })

    metric_rows.append(
        m
    )

standard_metrics = pd.DataFrame(
    metric_rows
)

random_metric_rows = []

for (
    camera,
    rep
), g in random_oof.groupby(
    [
        "camera_type",
        "rep",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "model":
            "random_other_multiband_delta_snv",

        "rep":
            int(
                rep
            ),
    })

    random_metric_rows.append(
        m
    )

random_metrics = pd.DataFrame(
    random_metric_rows
)

random_summary = (
    random_metrics.groupby(
        "camera_type"
    )[
        [
            "fruit_balanced_mae",
            "r2",
            "within_1_day_accuracy",
        ]
    ]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max",
        ]
    )
)

display(
    standard_metrics.sort_values(
        [
            "camera_type",
            "fruit_balanced_mae",
        ]
    )
)

display(
    random_summary
)

standard_metrics.to_csv(
    TABLES /
    "02_standard_baseline_complexity_metrics.csv",
    index=False
)

random_metrics.to_csv(
    TABLES /
    "03_random_other_baseline_metrics_by_rep.csv",
    index=False
)

selected_configs.to_csv(
    TABLES /
    "04_selected_configs.csv",
    index=False
)


# 16. Session-excluded — baseline propio, training mean y controles escalares

Para cada día objetivo:

- el modelo no observa ese día;
- el training-mean baseline sigue construyéndose solo a partir de día 0 de training;
- los controles escalares se calculan con baseline propio;
- se compara contra un `SessionExcluded_DummyMean`.


In [ ]:
def inner_select_self_session(
    camera,
    development_fruits,
    inner_assignment,
    feature_name
):

    best = None

    for alpha in RIDGE_ALPHAS:

        parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_fruits = (
                set(
                    development_fruits
                )
                -
                val_fruits
            )

            tr_all = make_self_features(
                camera,
                train_fruits
            )

            va_all = make_self_features(
                camera,
                val_fruits
            )

            if len(
                va_all[
                    "meta"
                ]
            ) == 0:
                continue

            for target_day in sorted(
                va_all[
                    "meta"
                ][
                    "storage_days_model"
                ].unique()
            ):

                tr_mask = (
                    tr_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy()
                    !=
                    target_day
                )

                va_mask = (
                    va_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy()
                    ==
                    target_day
                )

                pred = fit_predict_ridge(
                    tr_all[
                        feature_name
                    ][
                        tr_mask
                    ],
                    tr_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    )[
                        tr_mask
                    ],
                    va_all[
                        feature_name
                    ][
                        va_mask
                    ],
                    alpha
                )

                parts.append(
                    pd.DataFrame({
                        "fruit_id":
                            va_all[
                                "meta"
                            ][
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            )[
                                va_mask
                            ],

                        "y_true":
                            va_all[
                                "meta"
                            ][
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            )[
                                va_mask
                            ],

                        "y_pred":
                            pred,
                    })
                )

        p = pd.concat(
            parts,
            ignore_index=True
        )

        rank = (
            fruit_balanced_mae(
                p
            ),
            day_balanced_mae(
                p
            ),
            float(
                mean_absolute_error(
                    p[
                        "y_true"
                    ],
                    p[
                        "y_pred"
                    ]
                )
            ),
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),
            }

    return best


def inner_select_trainmean_session(
    camera,
    development_fruits,
    inner_assignment
):

    best = None

    for alpha in RIDGE_ALPHAS:

        parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_fruits = (
                set(
                    development_fruits
                )
                -
                val_fruits
            )

            tr_all = make_trainmean_delta_features(
                camera,
                rows_fruits=train_fruits,
                baseline_training_fruits=train_fruits
            )

            va_all = make_trainmean_delta_features(
                camera,
                rows_fruits=val_fruits,
                baseline_training_fruits=train_fruits
            )

            if len(
                va_all[
                    "meta"
                ]
            ) == 0:
                continue

            for target_day in sorted(
                va_all[
                    "meta"
                ][
                    "storage_days_model"
                ].unique()
            ):

                tr_mask = (
                    tr_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy()
                    !=
                    target_day
                )

                va_mask = (
                    va_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy()
                    ==
                    target_day
                )

                pred = fit_predict_ridge(
                    tr_all[
                        "X"
                    ][
                        tr_mask
                    ],
                    tr_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    )[
                        tr_mask
                    ],
                    va_all[
                        "X"
                    ][
                        va_mask
                    ],
                    alpha
                )

                parts.append(
                    pd.DataFrame({
                        "fruit_id":
                            va_all[
                                "meta"
                            ][
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            )[
                                va_mask
                            ],

                        "y_true":
                            va_all[
                                "meta"
                            ][
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            )[
                                va_mask
                            ],

                        "y_pred":
                            pred,
                    })
                )

        p = pd.concat(
            parts,
            ignore_index=True
        )

        rank = (
            fruit_balanced_mae(
                p
            ),
            day_balanced_mae(
                p
            ),
            float(
                mean_absolute_error(
                    p[
                        "y_true"
                    ],
                    p[
                        "y_pred"
                    ]
                )
            ),
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),
            }

    return best


# 17. OOF session-excluded


In [ ]:
session_parts = []

for camera in CAMERAS:

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer.loc[
                outer[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            all_fruits
            -
            test_fruits
        )

        inner_assignment = inner[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold
        ][
            [
                "fruit_id",
                "inner_fold",
            ]
        ].copy()

        for model_name, feature_name in self_feature_map.items():

            best = inner_select_self_session(
                camera,
                development_fruits,
                inner_assignment,
                feature_name
            )

            tr_all = make_self_features(
                camera,
                development_fruits
            )

            te_all = make_self_features(
                camera,
                test_fruits
            )

            for target_day in sorted(
                te_all[
                    "meta"
                ][
                    "storage_days_model"
                ].unique()
            ):

                tr_mask = (
                    tr_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy()
                    !=
                    target_day
                )

                te_mask = (
                    te_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy()
                    ==
                    target_day
                )

                pred = fit_predict_ridge(
                    tr_all[
                        feature_name
                    ][
                        tr_mask
                    ],
                    tr_all[
                        "meta"
                    ][
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    )[
                        tr_mask
                    ],
                    te_all[
                        feature_name
                    ][
                        te_mask
                    ],
                    best[
                        "alpha"
                    ]
                )

                session_parts.append(
                    pd.DataFrame({
                        "camera_type":
                            camera,

                        "protocol":
                            "session_excluded",

                        "model":
                            model_name,

                        "outer_fold":
                            int(
                                outer_fold
                            ),

                        "alpha":
                            float(
                                best[
                                    "alpha"
                                ]
                            ),

                        "fruit_id":
                            te_all[
                                "meta"
                            ][
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            )[
                                te_mask
                            ],

                        "storage_days":
                            te_all[
                                "meta"
                            ][
                                "storage_days_model"
                            ].to_numpy(
                                dtype=int
                            )[
                                te_mask
                            ],

                        "y_true":
                            te_all[
                                "meta"
                            ][
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            )[
                                te_mask
                            ],

                        "y_pred":
                            pred,
                    })
                )

        best_mean = inner_select_trainmean_session(
            camera,
            development_fruits,
            inner_assignment
        )

        trm = make_trainmean_delta_features(
            camera,
            rows_fruits=development_fruits,
            baseline_training_fruits=development_fruits
        )

        tem = make_trainmean_delta_features(
            camera,
            rows_fruits=test_fruits,
            baseline_training_fruits=development_fruits
        )

        for target_day in sorted(
            tem[
                "meta"
            ][
                "storage_days_model"
            ].unique()
        ):

            tr_mask = (
                trm[
                    "meta"
                ][
                    "storage_days_model"
                ].to_numpy()
                !=
                target_day
            )

            te_mask = (
                tem[
                    "meta"
                ][
                    "storage_days_model"
                ].to_numpy()
                ==
                target_day
            )

            pred = fit_predict_ridge(
                trm[
                    "X"
                ][
                    tr_mask
                ],
                trm[
                    "meta"
                ][
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                )[
                    tr_mask
                ],
                tem[
                    "X"
                ][
                    te_mask
                ],
                best_mean[
                    "alpha"
                ]
            )

            session_parts.append(
                pd.DataFrame({
                    "camera_type":
                        camera,

                    "protocol":
                        "session_excluded",

                    "model":
                        "training_mean_multiband_delta_snv",

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "alpha":
                        float(
                            best_mean[
                                "alpha"
                            ]
                        ),

                    "fruit_id":
                        tem[
                            "meta"
                        ][
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        )[
                            te_mask
                        ],

                    "storage_days":
                        tem[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        )[
                            te_mask
                        ],

                    "y_true":
                        tem[
                            "meta"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        )[
                            te_mask
                        ],

                    "y_pred":
                        pred,
                })
            )

session_oof = pd.concat(
    session_parts,
    ignore_index=True
)

for camera in CAMERAS:

    for model_name in list(
        self_feature_map.keys()
    ) + [
        "training_mean_multiband_delta_snv"
    ]:

        g = session_oof[
            (
                session_oof[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                session_oof[
                    "model"
                ]
                ==
                model_name
            )
        ]

        assert len(
            g
        ) == 159

print("✅ Session-excluded completo.")


# 18. Null correcto session-excluded


In [ ]:
session_null_parts = []

reference = later_tables[
    "VIS_COR"
][
    [
        "fruit_id",
        "storage_days_model",
    ]
].copy()

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    dev = reference[
        ~reference[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    te_all = reference[
        reference[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    for target_day in sorted(
        te_all[
            "storage_days_model"
        ].unique()
    ):

        tr = dev[
            dev[
                "storage_days_model"
            ]
            !=
            target_day
        ]

        te = te_all[
            te_all[
                "storage_days_model"
            ]
            ==
            target_day
        ]

        null_value = float(
            tr[
                "storage_days_model"
            ].mean()
        )

        session_null_parts.append(
            pd.DataFrame({
                "outer_fold":
                    int(
                        outer_fold
                    ),

                "fruit_id":
                    te[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    np.repeat(
                        null_value,
                        len(
                            te
                        )
                    ),
            })
        )

session_null = pd.concat(
    session_null_parts,
    ignore_index=True
)

session_null_metrics = metric_bundle(
    session_null
)

print(
    "SessionExcluded_DummyMean FB-MAE:",
    session_null_metrics[
        "fruit_balanced_mae"
    ]
)


# 19. Métricas session-excluded


In [ ]:
session_metric_rows = []

for (
    camera,
    model
), g in session_oof.groupby(
    [
        "camera_type",
        "model",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "model":
            model,

        "protocol":
            "session_excluded",
    })

    session_metric_rows.append(
        m
    )

session_metrics = pd.DataFrame(
    session_metric_rows
)

display(
    session_metrics.sort_values(
        [
            "camera_type",
            "fruit_balanced_mae",
        ]
    )
)

session_metrics.to_csv(
    TABLES /
    "05_session_excluded_baseline_complexity_metrics.csv",
    index=False
)


# 20. Bootstrap pareado por fruto

Comparaciones centrales para cada cámara:

- self multiband vs training-mean multiband;
- self multiband vs RMS;
- self multiband vs spectral angle;
- self multiband vs RMS+angle;
- self multiband vs DummyMedian;
- session-excluded self multiband vs training-mean;
- session-excluded self multiband vs null correcto.

Para random-other se compara el promedio por fruto sobre las 10 repeticiones.


In [ ]:
def per_fruit_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return (
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
    )


def paired_bootstrap(
    a,
    b,
    label,
    n_boot=5000,
    seed=20260818
):

    ids = sorted(
        set(
            a.index
        )
        &
        set(
            b.index
        )
    )

    av = a.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    bv = b.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    diff = (
        av
        -
        bv
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    for i in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            len(
                diff
            ),
            size=len(
                diff
            )
        )

        boot[
            i
        ] = np.mean(
            diff[
                idx
            ]
        )

    return {
        "comparison":
            label,

        "n_fruits":
            int(
                len(
                    diff
                )
            ),

        "mean_delta_mae_days":
            float(
                diff.mean()
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),

        "fraction_first_better":
            float(
                np.mean(
                    diff
                    <
                    0
                )
            ),
    }


bootstrap_rows = []
seed_offset = 0

standard_null_pf = per_fruit_mae(
    standard_null
)

session_null_pf = per_fruit_mae(
    session_null
)

for camera in CAMERAS:

    self_std = standard_oof[
        (
            standard_oof[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            standard_oof[
                "model"
            ]
            ==
            "self_multiband_delta_snv"
        )
    ]

    self_pf = per_fruit_mae(
        self_std
    )

    comparison_models = [
        "training_mean_multiband_delta_snv",
        "self_scalar_rms",
        "self_scalar_angle",
        "self_scalar_rms_angle",
    ]

    for other_model in comparison_models:

        other = standard_oof[
            (
                standard_oof[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                standard_oof[
                    "model"
                ]
                ==
                other_model
            )
        ]

        bootstrap_rows.append(
            paired_bootstrap(
                self_pf,
                per_fruit_mae(
                    other
                ),
                (
                    f"{camera} self_multiband minus "
                    f"{other_model} standard"
                ),
                BOOTSTRAP_REPS,
                RANDOM_SEED + seed_offset
            )
        )

        seed_offset += 1

    bootstrap_rows.append(
        paired_bootstrap(
            self_pf,
            standard_null_pf,
            (
                f"{camera} self_multiband minus "
                "DummyMedian standard"
            ),
            BOOTSTRAP_REPS,
            RANDOM_SEED + seed_offset
        )
    )

    seed_offset += 1

    # Random-other: promedio de error por fruto sobre las repeticiones.
    rg = random_oof[
        random_oof[
            "camera_type"
        ]
        ==
        camera
    ].copy()

    rg[
        "abs_error"
    ] = np.abs(
        rg[
            "y_true"
        ]
        -
        rg[
            "y_pred"
        ]
    )

    random_pf = (
        rg.groupby(
            [
                "rep",
                "fruit_id",
            ]
        )[
            "abs_error"
        ]
        .mean()
        .groupby(
            "fruit_id"
        )
        .mean()
    )

    bootstrap_rows.append(
        paired_bootstrap(
            self_pf,
            random_pf,
            (
                f"{camera} self_multiband minus "
                "random-other mean standard"
            ),
            BOOTSTRAP_REPS,
            RANDOM_SEED + seed_offset
        )
    )

    seed_offset += 1

    self_sess = session_oof[
        (
            session_oof[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            session_oof[
                "model"
            ]
            ==
            "self_multiband_delta_snv"
        )
    ]

    mean_sess = session_oof[
        (
            session_oof[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            session_oof[
                "model"
            ]
            ==
            "training_mean_multiband_delta_snv"
        )
    ]

    bootstrap_rows.append(
        paired_bootstrap(
            per_fruit_mae(
                self_sess
            ),
            per_fruit_mae(
                mean_sess
            ),
            (
                f"{camera} self_multiband minus "
                "training_mean session-excluded"
            ),
            BOOTSTRAP_REPS,
            RANDOM_SEED + seed_offset
        )
    )

    seed_offset += 1

    bootstrap_rows.append(
        paired_bootstrap(
            per_fruit_mae(
                self_sess
            ),
            session_null_pf,
            (
                f"{camera} self_multiband minus "
                "correct null session-excluded"
            ),
            BOOTSTRAP_REPS,
            RANDOM_SEED + seed_offset
        )
    )

    seed_offset += 1

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES /
    "06_paired_fruit_bootstrap_baseline_complexity.csv",
    index=False
)


# 21. Error por día de los controles principales


In [ ]:
day_rows = []

for protocol_name, pred_df in [
    (
        "standard",
        standard_oof
    ),
    (
        "session_excluded",
        session_oof
    ),
]:

    keep_models = [
        "self_multiband_delta_snv",
        "training_mean_multiband_delta_snv",
        "self_scalar_rms",
        "self_scalar_angle",
        "self_scalar_rms_angle",
    ]

    z = pred_df[
        pred_df[
            "model"
        ].isin(
            keep_models
        )
    ]

    for (
        camera,
        model,
        day
    ), g in z.groupby(
        [
            "camera_type",
            "model",
            "storage_days",
        ]
    ):

        err = (
            g[
                "y_pred"
            ].to_numpy(
                dtype=float
            )
            -
            g[
                "y_true"
            ].to_numpy(
                dtype=float
            )
        )

        day_rows.append({
            "protocol":
                protocol_name,

            "camera_type":
                camera,

            "model":
                model,

            "storage_days":
                int(
                    day
                ),

            "n":
                int(
                    len(
                        g
                    )
                ),

            "mae":
                float(
                    np.mean(
                        np.abs(
                            err
                        )
                    )
                ),

            "bias":
                float(
                    np.mean(
                        err
                    )
                ),
        })

day_metrics = pd.DataFrame(
    day_rows
)

day_metrics.to_csv(
    TABLES /
    "07_error_bias_by_day.csv",
    index=False
)

display(
    day_metrics[
        day_metrics[
            "camera_type"
        ]
        ==
        "VIS_COR"
    ]
)


# 22. Figura principal — especificidad del baseline


In [ ]:
for camera in CAMERAS:

    std = standard_metrics[
        standard_metrics[
            "camera_type"
        ]
        ==
        camera
    ]

    names = [
        "self_multiband_delta_snv",
        "training_mean_multiband_delta_snv",
    ]

    values = [
        float(
            std.loc[
                std[
                    "model"
                ]
                ==
                name,
                "fruit_balanced_mae"
            ].iloc[
                0
            ]
        )
        for name in names
    ]

    random_mean = float(
        random_metrics.loc[
            random_metrics[
                "camera_type"
            ]
            ==
            camera,
            "fruit_balanced_mae"
        ].mean()
    )

    labels = [
        "Self baseline",
        "Training-mean baseline",
        "Random-other baseline",
    ]

    values.append(
        random_mean
    )

    fig, ax = plt.subplots(
        figsize=(8, 4.8)
    )

    ax.bar(
        labels,
        values
    )

    ax.axhline(
        standard_null_metrics[
            "fruit_balanced_mae"
        ],
        linestyle="--",
        label="DummyMedian"
    )

    ax.set_ylabel(
        "MAE fruit-balanced (días)"
    )

    ax.set_title(
        f"{camera} — especificidad del baseline"
    )

    ax.tick_params(
        axis="x",
        rotation=15
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"01_baseline_specificity_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 23. Figura principal — complejidad de la trayectoria


In [ ]:
for camera in CAMERAS:

    std = standard_metrics[
        standard_metrics[
            "camera_type"
        ]
        ==
        camera
    ]

    order = [
        "self_multiband_delta_snv",
        "self_scalar_rms",
        "self_scalar_angle",
        "self_scalar_rms_angle",
    ]

    plot_df = (
        std.set_index(
            "model"
        )
        .loc[
            order
        ]
        .reset_index()
    )

    fig, ax = plt.subplots(
        figsize=(8.5, 4.8)
    )

    ax.bar(
        [
            "Multiband Δ-SNV",
            "RMS",
            "Spectral angle",
            "RMS + angle",
        ],
        plot_df[
            "fruit_balanced_mae"
        ]
    )

    ax.axhline(
        standard_null_metrics[
            "fruit_balanced_mae"
        ],
        linestyle="--",
        label="DummyMedian"
    )

    ax.set_ylabel(
        "MAE fruit-balanced (días)"
    )

    ax.set_title(
        f"{camera} — multibanda vs distancia al baseline"
    )

    ax.tick_params(
        axis="x",
        rotation=15
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"02_trajectory_complexity_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 24. Tabla maestra de conclusiones cuantitativas


In [ ]:
master_rows = []

for _, r in standard_metrics.iterrows():

    z = r.to_dict()

    z[
        "comparison_family"
    ] = "standard"

    master_rows.append(
        z
    )

for _, r in session_metrics.iterrows():

    z = r.to_dict()

    z[
        "comparison_family"
    ] = "session_excluded"

    master_rows.append(
        z
    )

master_metrics = pd.DataFrame(
    master_rows
)

display(
    master_metrics.sort_values(
        [
            "camera_type",
            "comparison_family",
            "fruit_balanced_mae",
        ]
    )
)

master_metrics.to_csv(
    TABLES /
    "08_master_baseline_complexity_metrics.csv",
    index=False
)


# 25. Guardar resultados centrales


In [ ]:
STANDARD_OOF_PATH = (
    RESULTS /
    "kaki_baseline_complexity_standard_oof.csv"
)

RANDOM_OOF_PATH = (
    RESULTS /
    "kaki_random_other_baseline_oof.csv"
)

SESSION_OOF_PATH_OUT = (
    RESULTS /
    "kaki_baseline_complexity_session_excluded_oof.csv"
)

STANDARD_NULL_PATH = (
    RESULTS /
    "kaki_standard_dummy_median_days_gt0_oof.csv"
)

SESSION_NULL_PATH = (
    RESULTS /
    "kaki_session_excluded_dummy_mean_days_gt0_oof.csv"
)

MASTER_METRICS_PATH = (
    RESULTS /
    "kaki_baseline_complexity_master_metrics.csv"
)

standard_oof.to_csv(
    STANDARD_OOF_PATH,
    index=False
)

random_oof.to_csv(
    RANDOM_OOF_PATH,
    index=False
)

session_oof.to_csv(
    SESSION_OOF_PATH_OUT,
    index=False
)

standard_null.to_csv(
    STANDARD_NULL_PATH,
    index=False
)

session_null.to_csv(
    SESSION_NULL_PATH,
    index=False
)

master_metrics.to_csv(
    MASTER_METRICS_PATH,
    index=False
)

print("✅ Resultados guardados.")


# 26. Resumen científico automático


In [ ]:
def json_safe(
    obj
):

    if isinstance(
        obj,
        dict
    ):
        return {
            str(k):
                json_safe(
                    v
                )
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        (
            list,
            tuple,
        )
    ):
        return [
            json_safe(
                v
            )
            for v in obj
        ]

    if isinstance(
        obj,
        np.integer
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating
    ):
        return (
            None
            if np.isnan(
                obj
            )
            else float(
                obj
            )
        )

    if isinstance(
        obj,
        np.bool_
    ):
        return bool(
            obj
        )

    if isinstance(
        obj,
        np.ndarray
    ):
        return json_safe(
            obj.tolist()
        )

    return obj


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "task": {
        "target":
            "storage_days_model",

        "days":
            [
                1,
                2,
                3,
                4,
                6,
                7,
                8,
            ],

        "n_fruit_days":
            159,

        "n_fruits":
            38,

        "baseline_day0_required_for_self_models":
            True,
    },

    "random_other_baseline_repetitions":
        N_RANDOM_BASELINE_REPS,

    "standard_metrics":
        standard_metrics.to_dict(
            orient="records"
        ),

    "random_other_metrics_by_rep":
        random_metrics.to_dict(
            orient="records"
        ),

    "session_excluded_metrics":
        session_metrics.to_dict(
            orient="records"
        ),

    "standard_null":
        standard_null_metrics,

    "session_excluded_null":
        session_null_metrics,

    "paired_bootstrap":
        bootstrap_results.to_dict(
            orient="records"
        ),

    "interpretation_rules": [
        (
            "If self baseline outperforms training-mean and random-other baselines, "
            "the fruit-specific longitudinal reference contributes information."
        ),
        (
            "If multiband delta outperforms RMS, spectral angle, and their combination, "
            "the trajectory contains useful spectral-shape information beyond simple "
            "distance from baseline."
        ),
        (
            "Random-other baseline is a negative control and is not tuned independently."
        ),
    ],

    "limitations": [
        (
            "Self-baseline models require a day-0 measurement of the same fruit."
        ),
        (
            "Random-other control uses a finite number of deterministic permutations."
        ),
        (
            "Session-excluded testing remains a stress test, not causal deconfounding."
        ),
        (
            "All data come from one acquisition campaign; external validation is absent."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_baseline_specificity_trajectory_complexity_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 27. Notas automáticas de revisión


In [ ]:
review_lines = [
    "KAKI — BASELINE SPECIFICITY & TRAJECTORY COMPLEXITY",
    "",
    "Task: storage_days > 0",
    "N = 159 fruit-days from 38 fruits",
    "",
    (
        "Standard null FB-MAE = "
        f"{standard_null_metrics['fruit_balanced_mae']:.4f} d"
    ),
    (
        "Session-excluded null FB-MAE = "
        f"{session_null_metrics['fruit_balanced_mae']:.4f} d"
    ),
    "",
]

for camera in CAMERAS:

    review_lines.append(
        f"{camera} — STANDARD:"
    )

    g = standard_metrics[
        standard_metrics[
            "camera_type"
        ]
        ==
        camera
    ].sort_values(
        "fruit_balanced_mae"
    )

    for _, r in g.iterrows():

        review_lines.append(
            (
                f"  {r['model']}: "
                f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
                f"R2={r['r2']:.4f}, "
                f"±1d={r['within_1_day_accuracy']:.4f}"
            )
        )

    rm = random_metrics[
        random_metrics[
            "camera_type"
        ]
        ==
        camera
    ]

    review_lines.append(
        (
            "  random_other_multiband_delta_snv: "
            f"mean FB-MAE={rm['fruit_balanced_mae'].mean():.4f} "
            f"± {rm['fruit_balanced_mae'].std():.4f}"
        )
    )

    review_lines.append(
        ""
    )

    review_lines.append(
        f"{camera} — SESSION-EXCLUDED:"
    )

    gs = session_metrics[
        session_metrics[
            "camera_type"
        ]
        ==
        camera
    ].sort_values(
        "fruit_balanced_mae"
    )

    for _, r in gs.iterrows():

        review_lines.append(
            (
                f"  {r['model']}: "
                f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
                f"R2={r['r2']:.4f}"
            )
        )

    review_lines.append(
        ""
    )

review_lines.extend([
    "Interpretation:",
    (
        "The central test is whether the same-fruit multiband baseline-referenced "
        "trajectory beats both non-specific baselines and scalar distance controls."
    ),
    (
        "If it does, stable fruit identity and simple distance-from-baseline are "
        "insufficient explanations for the temporal signal."
    ),
    "",
    (
        "After reviewing this package, decide whether the main classical-model "
        "phase can be frozen for manuscript synthesis."
    ),
])

REVIEW_PATH = (
    RESULTS /
    "BASELINE_COMPLEXITY_REVIEW_NOTES.txt"
)

REVIEW_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_PATH.read_text()
)
